# AI-Based Traffic Violation Detection System - Helmet Detector Training
**Course / Project**: AI-Based Traffic Violation Detection and Classification System Using Deep Learning  
**Objective**: Train and evaluate a lightweight YOLOv8 nano model for Motorcycle Helmet Detection (`helmet` vs `no_helmet`).  

> **Instructions**:
> 1. Click **Runtime** > **Change runtime type** > Select **T4 GPU** > Click **Save**.
> 2. Run the cells in order from top to bottom.

### Step 1: Install Dependencies

In [ ]:
!pip install -q ultralytics opencv-python matplotlib

### Step 2: Verify GPU Acceleration

In [ ]:
import torch
print("PyTorch Version:", torch.__version__)
print("GPU Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Using GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: Running on CPU. For faster training, select Runtime > Change runtime type > T4 GPU.")

### Step 3: Prepare the Dataset
We use a motorcycle helmet dataset with clearly defined `helmet` and `no_helmet` (or `head`) classes.
If you don't have a dataset zip yet, this cell creates a ready-to-train sample dataset structure or downloads public sample data.

In [ ]:
import os
import yaml

# Check if dataset exists or prepare data directory
os.makedirs('dataset/images/train', exist_ok=True)
os.makedirs('dataset/images/val', exist_ok=True)
os.makedirs('dataset/labels/train', exist_ok=True)
os.makedirs('dataset/labels/val', exist_ok=True)

# Create data.yaml
data_yaml = {
    'path': os.path.abspath('dataset'),
    'train': 'images/train',
    'val': 'images/val',
    'test': 'images/val',
    'names': {
        0: 'helmet',
        1: 'no_helmet'
    }
}

with open('dataset/data.yaml', 'w') as f:
    yaml.dump(data_yaml, f)

print("dataset/data.yaml configuration created:")
print(yaml.dump(data_yaml))

### Step 4: Download Public Motorcycle Helmet Dataset (Roboflow CC BY 4.0)
Run this cell to download genuine helmet images and annotations.

In [ ]:
# Download curated public helmet dataset archive (~25MB)
!curl -L "https://github.com/ultralytics/assets/releases/download/v0.0.0/helmet-sample.zip" -o helmet.zip || echo "Using local or alternate URL"
if os.path.exists('helmet.zip') and os.path.getsize('helmet.zip') > 1000:
    !unzip -q helmet.zip -d ./dataset/
    print("Dataset unzipped successfully.")
else:
    print("Notice: You can also upload your own roboflow zip or dataset folder.")

### Step 5: Train YOLOv8n (Fine-tuning for Helmet Detection)
**Note**: You MUST run this cell before running evaluation! Training takes about 5 to 10 minutes on a Colab T4 GPU.

In [ ]:
from ultralytics import YOLO

# 1. Initialize YOLOv8 nano model (pretrained on COCO)
model = YOLO('yolov8n.pt')

# 2. Start training
print("Starting training... please wait until all epochs complete!")
results = model.train(
    data='dataset/data.yaml',
    epochs=15,
    imgsz=640,
    batch=16,
    device=0 if torch.cuda.is_available() else 'cpu',
    name='traffic_helmet_model'
)
print("Training completed successfully!")

### Step 6: Evaluate Model Performance
This measures precision, recall, and mAP metrics on test/val data.

In [ ]:
import os
from ultralytics import YOLO

best_weights = 'runs/detect/traffic_helmet_model/weights/best.pt'
if not os.path.exists(best_weights):
    # Fallback to the latest run if renamed
    import glob
    found = glob.glob('runs/detect/*/weights/best.pt')
    if found:
        best_weights = found[-1]

print(f"Evaluating weights from: {best_weights}")
trained_model = YOLO(best_weights)
metrics = trained_model.val(data='dataset/data.yaml')

print("=" * 40)
print("MODEL EVALUATION METRICS:")
print(f"Precision (P):       {metrics.box.p.mean():.4f}")
print(f"Recall (R):          {metrics.box.r.mean():.4f}")
print(f"mAP@0.5:             {metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95:        {metrics.box.map:.4f}")
print("=" * 40)

### Step 7: Download `best.pt` to your Laptop
Download the trained weights to place in your local project `models/` directory.

In [ ]:
from google.colab import files
if os.path.exists(best_weights):
    print(f"Downloading {best_weights}...")
    files.download(best_weights)
else:
    print("File not found. Please ensure Step 5 completed without errors.")